# Mental Health Clinical Trials Pipeline

How is mental health treatment research changing? This notebook downloads interventional trials from the ClinicalTrials.gov API (start dates 2010–2025), cleans them into four tables, groups conditions into eight DSM-5-aligned groups, flags psychedelic and ketamine treatments, and answers five questions with SQL.

**Parts:** A. Setup · B. Download · C. Clean and group · D. SQL analysis · E. Validation

Run the cells in order. Results refer to the download of 1 October 2026; see the README for methods and caveats.

## A. Setup
Mounts Google Drive and creates the project folders.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = '/content/drive/MyDrive/mental-health-trials-pipeline'
for sub in ['data/raw', 'data/clean', 'data/validation', 'sql', 'results']:
    os.makedirs(f'{PROJECT}/{sub}', exist_ok=True)
print(os.listdir(PROJECT))

## B. Download from the ClinicalTrials.gov API

**B1.** Test the API with one study (status 200 means the request worked).

In [ ]:
import requests, json, time

BASE = "https://clinicaltrials.gov/api/v2/studies"
test = requests.get(BASE, params={
    "query.cond": "depression",
    "pageSize": 1,
    "countTotal": "true",
}, timeout=60)

print("Status:", test.status_code)
data = test.json()
print("Total depression studies:", data.get("totalCount"))
print("Sections in one study:", list(data["studies"][0]["protocolSection"].keys()))

**B2.** Download every matching interventional study, 1,000 per page, and save the raw JSON to Drive.

In [ ]:
import os

# The analysis uses the download from 1 October 2026. If that file is already in your Drive,
# it is reused; set REDOWNLOAD = True to pull fresh data (counts will differ slightly).
REDOWNLOAD = False
RAW = f"{PROJECT}/data/raw/trials_raw.json"
download = REDOWNLOAD or not os.path.exists(RAW)

params = {
    "query.cond": ('depression OR anxiety OR PTSD OR "bipolar disorder" '
               'OR schizophrenia OR psychosis OR "substance use disorder" '
               'OR "obsessive-compulsive disorder" OR "eating disorder" '
               'OR "anorexia nervosa" OR "bulimia nervosa" OR "binge eating disorder"'),
    "filter.advanced": ("AREA[StudyType]INTERVENTIONAL AND "
                        "AREA[StartDate]RANGE[2010-01-01,2025-12-31]"),
    "pageSize": 1000,
    "countTotal": "true",
    "format": "json",
}

studies, token = [], None
while download:
    if token:
        params["pageToken"] = token
    r = requests.get(BASE, params=params, timeout=120)
    r.raise_for_status()
    page = r.json()
    if "totalCount" in page:
        print("Studies matching the search:", page["totalCount"])
    studies += page["studies"]
    token = page.get("nextPageToken")
    print(len(studies), "downloaded")
    if not token:
        break
    time.sleep(0.3)   # be polite to the server

if download:
    with open(RAW, "w") as f:
        json.dump(studies, f)
    print("Saved", len(studies), "studies")
else:
    print("Using the existing download:", RAW)

## C. Clean and group

**C1.** Flatten the nested JSON into four tables: trials, conditions, interventions and sites (one row per trial and country).

In [ ]:
import pandas as pd

def g(d, *keys, default=None):
    for k in keys:
        if not isinstance(d, dict) or k not in d:
            return default
        d = d[k]
    return d

with open(f"{PROJECT}/data/raw/trials_raw.json") as f:
    studies = json.load(f)

trials, conds, ints, sites = [], [], [], []
for s in studies:
    p = s.get("protocolSection", {})
    nct = g(p, "identificationModule", "nctId")
    start = g(p, "statusModule", "startDateStruct", "date")
    trials.append({
        "nct_id": nct,
        "title": g(p, "identificationModule", "briefTitle"),
        "start_date": start,
        "start_year": int(start[:4]) if start else None,
        "overall_status": g(p, "statusModule", "overallStatus"),
        "why_stopped": g(p, "statusModule", "whyStopped"),
        "phase": "/".join(g(p, "designModule", "phases", default=[]) or []) or "NA",
        "enrollment": g(p, "designModule", "enrollmentInfo", "count"),
        "sponsor_name": g(p, "sponsorCollaboratorsModule", "leadSponsor", "name"),
        "sponsor_class": g(p, "sponsorCollaboratorsModule", "leadSponsor", "class"),
    })
    for c in g(p, "conditionsModule", "conditions", default=[]) or []:
        conds.append({"nct_id": nct, "condition": c})
    for i in g(p, "armsInterventionsModule", "interventions", default=[]) or []:
        ints.append({"nct_id": nct,
                     "intervention_type": i.get("type"),
                     "intervention_name": i.get("name")})
    locations = g(p, "contactsLocationsModule", "locations", default=[]) or []
    for country in {loc.get("country") for loc in locations if loc.get("country")}:
        sites.append({"nct_id": nct, "country": country})

trials = pd.DataFrame(trials)
conditions = pd.DataFrame(conds)
interventions = pd.DataFrame(ints)
sites = pd.DataFrame(sites)

for name, df in [("trials", trials), ("conditions", conditions),
                 ("interventions", interventions), ("sites", sites)]:
    print(f"{name:14s} {len(df):>8,} rows")
print(trials.head())

Quick checks: IDs are unique, start years are in range, and the status and phase mix looks sensible.

In [ ]:
print("IDs unique:", trials.nct_id.is_unique)
print("Start years:", trials.start_year.min(), "to", trials.start_year.max())
print(trials.overall_status.value_counts())
print(trials.phase.value_counts().head(8))

**C2.** Group each listed condition with keyword rules. Exclusions are checked first, then the first matching group wins; anything else is "Other".

In [ ]:
import re

EXCLUDE = (r"respiratory depression|cns depression"
           r"|(pre|post|peri)[- ]?operative anxiety|anxiety,? (pre|post|peri)[- ]?operative"
           r"|presurgical|pre-surgical|surgical anxiety|surgery anxiety"
           r"|dental anxiety|dental fear|procedural anxiety|kinesiophobia|traumatic brain injury")

GROUPS = {
    "Depression": r"depress|dysthym|\bmdd\b|anhedonia",
    "Anxiety": r"anxi|panic|phobi|\bgad\b|agoraphobi",
    "OCD": r"obsessive|\bocd\b",
    "PTSD": r"ptsd|post[- ]?traumatic|acute stress disorder",
    "Bipolar disorder": r"bipolar|\bmania\b|manic",
    "Schizophrenia & psychosis": r"schizo|psychosis|psychoses|psychotic",
    "Substance use": (r"substance|alcohol use|alcohol abuse|alcohol dependence|alcoholism"
                      r"|hazardous alcohol|heavy drinking|binge drinking"
                      r"|opioid|opiate|cocaine|methamphetamine|stimulant use|cannabis use"
                      r"|addict|use disorder|dependence"),
    "Eating disorders": (r"anorexia nervosa|bulimi|binge.?eating|eating disorder|\barfid\b"
                         r"|avoidant.?restrictive|disordered eating|feeding disorder,? (of )?infancy"),
}

def group_of(text):
    t = text.lower() if isinstance(text, str) else ""
    if re.search(EXCLUDE, t):
        return "Other"
    for name, pattern in GROUPS.items():
        if re.search(pattern, t):
            return name
    return "Other"

conditions["condition_group"] = conditions["condition"].map(group_of)
print(conditions["condition_group"].value_counts())

**C3.** Flag psychedelic/MDMA and ketamine/esketamine treatments by substance name and company drug code. Placebo arms named after a drug are excluded.

In [ ]:
PSYCHEDELIC = (
    # substances
    r"psilocybin|psilocin|mdma|midomafetamine|\blsd\b|lysergic"
    r"|\bdmt\b|dimethyltryptamine|5-meo|ayahuasca|ibogaine|noribogaine"
    r"|mescaline|salvinorin|psychedelic"
    # company drug codes
    r"|comp ?360"        # Compass Pathways, psilocybin
    r"|mm-?120"          # MindMed, LSD
    r"|cyb00[34]"        # Cybin, deuterated psilocybin analogue / DMT
    r"|gh00[12]"         # GH Research, 5-MeO-DMT
    r"|bpl-?003"         # Beckley Psytech, 5-MeO-DMT nasal spray
    r"|spl0?26"          # Small Pharma, DMT
    r"|re-?104"          # Reunion Neuroscience, psilocybin-like prodrug
    r"|bmnd0\d"          # Biomind Labs, 5-MeO-DMT
)
KETAMINE = r"ketamine|spravato"   # also matches esketamine and arketamine

def drug_class(name):
    n = name.lower() if isinstance(name, str) else ""
    if n.startswith("placebo") or "matched placebo" in n:
        return None
    if re.search(PSYCHEDELIC, n):
        return "Psychedelic / MDMA"
    if re.search(KETAMINE, n):
        return "Ketamine / esketamine"
    return None

interventions["psychedelic_class"] = interventions["intervention_name"].map(drug_class)
print(interventions["psychedelic_class"].value_counts())

**Analysis set.** A trial is in scope if it lists at least one grouped condition. Also shows how many flagged trials fall outside the scope.

In [ ]:
grouped_ids = set(conditions.loc[conditions.condition_group != "Other", "nct_id"])
trials["in_scope"] = trials["nct_id"].isin(grouped_ids).astype(int)
print(trials["in_scope"].value_counts())

flagged = interventions.dropna(subset=["psychedelic_class"])
summary = (flagged.assign(in_scope=flagged.nct_id.isin(grouped_ids))
                  .groupby(["psychedelic_class", "in_scope"])["nct_id"].nunique()
                  .unstack(fill_value=0))
print(summary)

**C4.** Save the clean tables as CSV and load them into SQLite. `q()` runs a SQL query and returns a DataFrame.

In [ ]:
import sqlite3

tables = {"trials": trials, "conditions": conditions,
          "interventions": interventions, "sites": sites}
for name, df in tables.items():
    df.to_csv(f"{PROJECT}/data/clean/{name}.csv", index=False)

con = sqlite3.connect(f"{PROJECT}/data/trials.db")
for name, df in tables.items():
    df.to_sql(name, con, if_exists="replace", index=False)

def q(sql):
    return pd.read_sql(sql, con)

print(q("SELECT name FROM sqlite_master WHERE type = 'table'"))

## D. SQL analysis

Two rules apply throughout: count `DISTINCT nct_id`, because joins repeat trials with several conditions, treatments or countries; and filter on `in_scope = 1` whenever a query doesn't already restrict to grouped conditions.

### Question 1: How has the number of trials changed, by condition?

**D1.** Trials started per year in each condition group.

In [ ]:
d1 = q("""
SELECT t.start_year, c.condition_group,
       COUNT(DISTINCT t.nct_id) AS trials
FROM trials t
JOIN conditions c ON t.nct_id = c.nct_id
WHERE c.condition_group <> 'Other'
GROUP BY t.start_year, c.condition_group
ORDER BY t.start_year, c.condition_group;
""")
print(d1.pivot(index="start_year", columns="condition_group", values="trials").fillna(0).astype(int).to_string())

Check: how many anxiety trials mention COVID-19 in the title? (Tests whether the pandemic explains the 2021 jump.)

In [ ]:
print(q("""
SELECT t.start_year,
       COUNT(DISTINCT t.nct_id) AS anxiety_trials,
       COUNT(DISTINCT CASE WHEN LOWER(t.title) LIKE '%covid%'
                            OR LOWER(t.title) LIKE '%pandemic%'
                            OR LOWER(t.title) LIKE '%coronavirus%'
                            OR LOWER(t.title) LIKE '%sars-cov%'
                           THEN t.nct_id END) AS covid_in_title
FROM trials t JOIN conditions c ON t.nct_id = c.nct_id
WHERE c.condition_group = 'Anxiety' AND t.start_year BETWEEN 2018 AND 2023
GROUP BY t.start_year ORDER BY t.start_year;
""").to_string())

**D2.** Year-over-year change with `LAG()`.

In [ ]:
d2 = q("""
WITH yearly AS (
  SELECT t.start_year, c.condition_group, COUNT(DISTINCT t.nct_id) AS trials
  FROM trials t
  JOIN conditions c ON t.nct_id = c.nct_id
  WHERE c.condition_group <> 'Other'
  GROUP BY t.start_year, c.condition_group
)
SELECT start_year, condition_group, trials,
       LAG(trials) OVER (PARTITION BY condition_group ORDER BY start_year) AS prev_year,
       ROUND(100.0 * (trials - LAG(trials) OVER (PARTITION BY condition_group ORDER BY start_year))
             / LAG(trials) OVER (PARTITION BY condition_group ORDER BY start_year), 1) AS pct_change
FROM yearly
ORDER BY condition_group, start_year;
""")
print(d2.pivot(index="start_year", columns="condition_group", values="pct_change").to_string())

## Findings so far

**D1: Trials per year by condition (2010–12 avg → 2023–25 avg)**
- Anxiety ×5.7, substance use ×3.5, eating disorders ×3.6, depression ×2.5, OCD and PTSD ×2.0
- Schizophrenia and bipolar disorder are nearly flat (×1.2): research on common disorders grew far faster than research on serious mental illness
- Registry growth inflates all counts, so compare groups with each other rather than citing raw growth
- No visible drop in 2024–25, so reporting lag isn't a problem

**COVID check (anxiety trials with COVID in the title)**
- 2020: 38 of 263 anxiety trials were COVID-related; non-COVID anxiety trials fell from 258 to 225
- 2021: anxiety rose to 415, but only 25 were COVID-related; non-COVID trials stepped up to ~390 and stayed there
- The 2021 rise is not explained by COVID-specific trials

**D2: Year-over-year change (LAG)**
- 2020: 7 of 8 groups fell (anxiety held only because of COVID trials)
- 2021: all 8 groups rose
- OCD, eating disorders and bipolar are too small for year-to-year % to mean much

### Question 2: Which treatment types are growing?

**D3.** Each treatment type's share of the year (a trial testing a drug plus therapy counts under both).

In [ ]:
d3 = q("""
WITH typed AS (
  SELECT DISTINCT t.nct_id, t.start_year, i.intervention_type
  FROM trials t
  JOIN interventions i ON t.nct_id = i.nct_id
  WHERE t.in_scope = 1
)
SELECT start_year, intervention_type, COUNT(*) AS trials,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY start_year), 1) AS pct_of_year
FROM typed
GROUP BY start_year, intervention_type
ORDER BY start_year, trials DESC;
""")
print(d3.pivot(index="start_year", columns="intervention_type", values="pct_of_year").fillna(0).to_string())

Check: actual counts behind the shares, and what the OTHER and DEVICE categories contain.

In [ ]:
# 1. Actual trial counts by type, for a few sample years
print(d3.pivot(index="start_year", columns="intervention_type", values="trials")
        .fillna(0).astype(int)[["DRUG", "BEHAVIORAL", "DEVICE", "OTHER"]]
        .loc[[2010, 2015, 2020, 2025]].to_string())

# 2. What's inside OTHER and DEVICE in recent years
for kind in ["OTHER", "DEVICE"]:
    print(f"\n{kind}, 2023–2025:")
    print(q(f"""
    SELECT i.intervention_name, COUNT(DISTINCT t.nct_id) AS trials
    FROM trials t JOIN interventions i ON t.nct_id = i.nct_id
    WHERE t.in_scope = 1 AND t.start_year >= 2023 AND i.intervention_type = '{kind}'
    GROUP BY i.intervention_name ORDER BY trials DESC LIMIT 15;
    """).to_string())

**D3: Treatment types (in-scope trials)**
- Drug share fell from 44% (2010) to 15% (2025), but drug trial counts stayed roughly level (277 → 251); non-drug treatments grew around them
- Behavioural trials grew ~3.4× (203 → 691) and are the largest type every year
- Device trials grew ~10× (27 → 272), driven by brain stimulation (TMS, rTMS, iTBS, tDCS) and virtual reality; by 2024–25 device trials matched drug trials in share
- "Other" is mostly comparison arms (placebo, treatment as usual, waitlist) plus non-drug therapies labelled inconsistently, so it's excluded from the treatment-type chart and the behavioural count is likely an undercount

### Question 3: Who sponsors the research, and has that shifted?

**D4.** Top 5 lead sponsors in each condition group with `RANK()`.

In [ ]:
d4 = q("""
WITH counts AS (
  SELECT c.condition_group, t.sponsor_name, t.sponsor_class,
         COUNT(DISTINCT t.nct_id) AS trials
  FROM trials t
  JOIN conditions c ON t.nct_id = c.nct_id
  WHERE c.condition_group <> 'Other'
  GROUP BY c.condition_group, t.sponsor_name, t.sponsor_class
),
ranked AS (
  SELECT *, RANK() OVER (PARTITION BY condition_group ORDER BY trials DESC) AS rnk
  FROM counts
)
SELECT * FROM ranked WHERE rnk <= 5
ORDER BY condition_group, rnk;
""")
print(d4.to_string())

**D4: Top 5 lead sponsors per condition**
- 40 of 41 top-5 slots are academic, hospital or government; the only company is Otsuka (#1 in schizophrenia, 60 trials)
- The US VA dominates PTSD (185 trials, ~6× the next sponsor) and also leads substance use
- CAMH (Toronto) is top 5 in four groups: schizophrenia #2, substance use #4, depression #5, bipolar #5
- Massachusetts General Hospital leads depression (126) and appears in 5 of 8 groups
- Caveat: lead sponsors only; industry is spread across many companies, so per-sponsor rankings understate it

**D4b.** Sponsor type by year.

In [ ]:
d4b = q("""
SELECT t.start_year, t.sponsor_class, COUNT(*) AS trials,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY t.start_year), 1) AS pct_of_year
FROM trials t
WHERE t.in_scope = 1
GROUP BY t.start_year, t.sponsor_class
ORDER BY t.start_year, trials DESC;
""")
print(d4b.pivot(index="start_year", columns="sponsor_class", values="pct_of_year").fillna(0).to_string())

Check: counts behind the shares.

In [ ]:
print(d4b.pivot(index="start_year", columns="sponsor_class", values="trials")
         .fillna(0).astype(int)[["INDUSTRY", "OTHER", "FED"]]
         .loc[[2010, 2015, 2020, 2025]].to_string())

**D4b: Sponsor type over time (in-scope trials)**
- Industry share fell from 24% (2010) to 9% (2025); most of the drop came by 2015, then flat at ~8–11%
- Universities and hospitals rose from 66% to 87%
- US federal (FED) share fell from 6% to 1%
- Fits the treatment-type trend: the fastest-growing types (behavioural, device) are mostly academic
- Counts: industry trials halved from 2010 to 2015 (134 → 69), then recovered to 126 by 2025; academic trials grew ~3.5× (366 → 1,269), so industry's *share* kept falling even as its numbers recovered
- The industry dip and recovery mirrors drug trials (277 in 2010 → 173 in 2020 → 251 in 2025); psychedelic trials are too few to explain the recovery

### Question 4: How often do trials stop early?

**D5.** Completed vs stopped early by sponsor type (ended trials only).

In [ ]:
d5 = q("""
SELECT sponsor_class,
       COUNT(*) AS ended_trials,
       SUM(overall_status = 'COMPLETED') AS completed,
       SUM(overall_status IN ('TERMINATED','WITHDRAWN','SUSPENDED')) AS stopped_early,
       ROUND(100.0 * SUM(overall_status IN ('TERMINATED','WITHDRAWN','SUSPENDED'))
             / COUNT(*), 1) AS pct_stopped_early
FROM trials
WHERE in_scope = 1
  AND overall_status IN ('COMPLETED','TERMINATED','WITHDRAWN','SUSPENDED')
GROUP BY sponsor_class
ORDER BY ended_trials DESC;
""")
print(d5.to_string())

print(q("""
SELECT why_stopped, COUNT(*) AS trials
FROM trials
WHERE in_scope = 1 AND why_stopped IS NOT NULL
GROUP BY why_stopped ORDER BY trials DESC LIMIT 20;
""").to_string())

**D5: Completed vs stopped early (ended trials)**
- Industry stops early most often: 15.9% vs 12.3% for universities/hospitals and 12.8% for US federal
- NIH, network and individual sponsors are too small to compare (<35 ended trials each)
- why_stopped is free text with many spellings of the same reason, plus blank strings; grouped into categories with CASE WHEN (D5b)

**D5b.** Why trials stop: free-text reasons grouped into categories with `CASE WHEN`.

In [ ]:
d5b = q("""
WITH reasons AS (
  SELECT sponsor_class, LOWER(why_stopped) AS r
  FROM trials
  WHERE in_scope = 1
    AND overall_status IN ('TERMINATED','WITHDRAWN','SUSPENDED')
    AND TRIM(COALESCE(why_stopped, '')) <> ''
),
grouped AS (
  SELECT sponsor_class,
    CASE
      WHEN r LIKE '%covid%' OR r LIKE '%pandemic%' OR r LIKE '%coronavirus%' THEN 'COVID-19'
      WHEN r LIKE '%efficacy%' OR r LIKE '%futility%' OR r LIKE '%interim analys%' THEN 'Efficacy / futility'
      WHEN r LIKE '%sponsor decision%' OR r LIKE '%business%' OR r LIKE '%strategic%'
        OR r LIKE '%portfolio%' OR r LIKE '%company decision%' OR r LIKE '%program%' THEN 'Sponsor / business decision'
      WHEN r LIKE '%fund%' OR r LIKE '%budget%' OR r LIKE '%financ%' THEN 'Funding'
      WHEN r LIKE '%recruit%' OR r LIKE '%enrol%' OR r LIKE '%accrual%'
        OR r LIKE '%participant%' OR r LIKE '%subject%' OR r LIKE '%patient%' THEN 'Recruitment'
      WHEN r LIKE '%pi %' OR r LIKE '%investigator%' OR r LIKE '%left the%'
        OR r LIKE '%staff%' OR r LIKE '%personnel%' THEN 'Investigator / staff'
      WHEN r LIKE '%safety%' OR r LIKE '%adverse%' THEN 'Safety'
      ELSE 'Other / unclear'
    END AS reason_group
  FROM reasons
)
SELECT reason_group,
       COUNT(*) AS trials,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct,
       SUM(sponsor_class = 'INDUSTRY') AS industry,
       SUM(sponsor_class = 'OTHER') AS academic
FROM grouped
GROUP BY reason_group
ORDER BY trials DESC;
""")
print(d5b.to_string())

Checks: the most common reasons overall, then what's left in "Other / unclear".

In [ ]:
print(q("""
SELECT LOWER(why_stopped) AS r, COUNT(*) AS n
FROM trials
WHERE in_scope = 1 AND overall_status IN ('TERMINATED','WITHDRAWN','SUSPENDED')
  AND TRIM(COALESCE(why_stopped,'')) <> ''
GROUP BY r ORDER BY n DESC LIMIT 25;
""").to_string())

In [ ]:
print(q("""
WITH reasons AS (
  SELECT LOWER(why_stopped) AS r
  FROM trials
  WHERE in_scope = 1
    AND overall_status IN ('TERMINATED','WITHDRAWN','SUSPENDED')
    AND TRIM(COALESCE(why_stopped, '')) <> ''
)
SELECT r, COUNT(*) AS n
FROM reasons
WHERE NOT (r LIKE '%covid%' OR r LIKE '%pandemic%' OR r LIKE '%coronavirus%'
        OR r LIKE '%efficacy%' OR r LIKE '%futility%' OR r LIKE '%interim analys%'
        OR r LIKE '%sponsor decision%' OR r LIKE '%business%' OR r LIKE '%strategic%'
        OR r LIKE '%portfolio%' OR r LIKE '%company decision%' OR r LIKE '%program%'
        OR r LIKE '%fund%' OR r LIKE '%budget%' OR r LIKE '%financ%'
        OR r LIKE '%recruit%' OR r LIKE '%enrol%' OR r LIKE '%accrual%'
        OR r LIKE '%participant%' OR r LIKE '%subject%' OR r LIKE '%patient%'
        OR r LIKE '%pi %' OR r LIKE '%investigator%' OR r LIKE '%left the%'
        OR r LIKE '%staff%' OR r LIKE '%personnel%'
        OR r LIKE '%safety%' OR r LIKE '%adverse%')
GROUP BY r ORDER BY n DESC LIMIT 25;
""").to_string())

**D5b: Why trials stop (1,212 stopped trials with a reason, grouped with CASE WHEN)**
- Overall: recruitment 27%, funding 16%, COVID 12%, investigator/staff 9%, sponsor decision 5%, efficacy/futility 4%, safety <1%
- Industry stops for strategy or results: business decision 27%, recruitment 21%, efficacy/futility 17%
- Academia stops for resources: recruitment 29%, funding 19%, COVID 14%, investigator/staff 10%
- Safety is almost never the stated reason (11 trials)
- ~25% of reasons are uncategorised; checked: a long tail of one-off wordings (none used more than twice), so no category is missing

### Question 5: Where does Canada fit?

**D6.** Share of trials in each condition group with at least one Canadian site.

In [ ]:
d6 = q("""
SELECT c.condition_group,
       COUNT(DISTINCT c.nct_id) AS all_trials,
       COUNT(DISTINCT CASE WHEN s.country = 'Canada' THEN c.nct_id END) AS with_canadian_site,
       ROUND(100.0 * COUNT(DISTINCT CASE WHEN s.country = 'Canada' THEN c.nct_id END)
             / COUNT(DISTINCT c.nct_id), 1) AS pct_canada
FROM conditions c
LEFT JOIN sites s ON c.nct_id = s.nct_id
WHERE c.condition_group <> 'Other'
GROUP BY c.condition_group
ORDER BY pct_canada DESC;
""")
print(d6.to_string())

print(q("""
SELECT COUNT(DISTINCT t.nct_id) AS trials_with_canadian_site
FROM trials t JOIN sites s ON t.nct_id = s.nct_id
WHERE t.in_scope = 1 AND s.country = 'Canada';
"""))

Checks: trials with no site listed, and the top 10 countries.

In [ ]:
# 1. How many in-scope trials list no site at all? (They count in the denominator above.)
print(q("""
SELECT COUNT(*) AS in_scope_trials,
       SUM(nct_id NOT IN (SELECT nct_id FROM sites)) AS no_site_listed
FROM trials WHERE in_scope = 1;
""").to_string())

# 2. Where does Canada rank among countries?
print(q("""
SELECT s.country, COUNT(DISTINCT t.nct_id) AS trials,
       ROUND(100.0 * COUNT(DISTINCT t.nct_id) / (SELECT COUNT(*) FROM trials WHERE in_scope = 1), 1) AS pct
FROM trials t JOIN sites s ON t.nct_id = s.nct_id
WHERE t.in_scope = 1
GROUP BY s.country ORDER BY trials DESC LIMIT 10;
""").to_string())

**D6: Canada (Question 5)**
- 1,020 of 14,713 in-scope trials (6.9%) have at least one Canadian site; 7.4% of trials that list any site
- Canada ranks #2 in ClinicalTrials.gov after the US (6,438; 43.8%), ahead of Turkey, China, France and the UK
- Per person, roughly 25 trials per million in Canada vs ~19 in the US
- Canadian share is fairly even across groups: highest in OCD (8.6%), bipolar (8.4%) and depression (8.2%), lowest in PTSD (5.3%, where the US VA dominates)
- Caveat: ClinicalTrials.gov is US-based; many European and Chinese trials register elsewhere, so country ranks reflect this registry
- 909 in-scope trials (6.2%) list no site, so percentages are slight underestimates

### Psychedelic and ketamine trials

**D7.** Trials per year with a running total (`SUM() OVER`).

In [ ]:
d7 = q("""
WITH yearly AS (
  SELECT t.start_year, i.psychedelic_class, COUNT(DISTINCT t.nct_id) AS trials
  FROM trials t
  JOIN interventions i ON t.nct_id = i.nct_id
  WHERE i.psychedelic_class IS NOT NULL AND t.in_scope = 1
  GROUP BY t.start_year, i.psychedelic_class
)
SELECT start_year, psychedelic_class, trials,
       SUM(trials) OVER (PARTITION BY psychedelic_class ORDER BY start_year) AS running_total
FROM yearly
ORDER BY psychedelic_class, start_year;
""")
print(d7.pivot(index="start_year", columns="psychedelic_class", values="trials").fillna(0).astype(int).to_string())
print(d7.pivot(index="start_year", columns="psychedelic_class", values="running_total").ffill().fillna(0).astype(int).to_string())

Sponsor type before and after 2021.

In [ ]:
print(q("""
SELECT i.psychedelic_class,
       CASE WHEN t.start_year >= 2021 THEN '2021-2025' ELSE '2010-2020' END AS period,
       t.sponsor_class,
       COUNT(DISTINCT t.nct_id) AS trials
FROM trials t JOIN interventions i ON t.nct_id = i.nct_id
WHERE i.psychedelic_class IS NOT NULL AND t.in_scope = 1
GROUP BY 1, 2, 3
ORDER BY 1, 2, trials DESC;
""").to_string())

**D7: Psychedelic and ketamine trials (in scope; running totals 318 ketamine, 165 psychedelic)**
- Psychedelic trials: 0–3 a year until 2017, 5–7 in 2018–20, then a surge from 2021 (19, 16, 29, 37, 31)
- 80% of psychedelic trials (132 of 165) started in 2021 or later
- Psychedelic trials outnumbered ketamine trials in 2023, 2024 and 2025
- Ketamine is steady at about 15–30 trials a year since 2012; no visible jump after esketamine's 2019 approval
- The 2021 surge was academic-led (university trials 17 → 95); industry's share of psychedelic trials fell from 48% to 23%
- Industry share of ketamine trials fell from 17% to 7% after esketamine's 2019 approval
- Industry is overrepresented in psychedelic research: it sponsors 23% of 2021–25 psychedelic trials (31 of 132), vs about 9% of all mental health trials; universities and hospitals still sponsor the majority (95)
- Psychedelic trials are too few to explain why industry-sponsored trials recovered overall (from 69 in 2015 to 126 in 2025)

### D8. Save every query and its result
Writes each query to `sql/` and its result to `results/`.

In [ ]:
import os
os.makedirs(f"{PROJECT}/sql", exist_ok=True)
os.makedirs(f"{PROJECT}/results", exist_ok=True)

QUERIES = {
"01_trials_per_year_by_condition": """
SELECT t.start_year, c.condition_group,
       COUNT(DISTINCT t.nct_id) AS trials
FROM trials t
JOIN conditions c ON t.nct_id = c.nct_id
WHERE c.condition_group <> 'Other'
GROUP BY t.start_year, c.condition_group
ORDER BY t.start_year, c.condition_group;
""",

"02_year_over_year_change": """
WITH yearly AS (
  SELECT t.start_year, c.condition_group, COUNT(DISTINCT t.nct_id) AS trials
  FROM trials t
  JOIN conditions c ON t.nct_id = c.nct_id
  WHERE c.condition_group <> 'Other'
  GROUP BY t.start_year, c.condition_group
)
SELECT start_year, condition_group, trials,
       LAG(trials) OVER (PARTITION BY condition_group ORDER BY start_year) AS prev_year,
       ROUND(100.0 * (trials - LAG(trials) OVER (PARTITION BY condition_group ORDER BY start_year))
             / LAG(trials) OVER (PARTITION BY condition_group ORDER BY start_year), 1) AS pct_change
FROM yearly
ORDER BY condition_group, start_year;
""",

"03_treatment_type_share": """
WITH typed AS (
  SELECT DISTINCT t.nct_id, t.start_year, i.intervention_type
  FROM trials t
  JOIN interventions i ON t.nct_id = i.nct_id
  WHERE t.in_scope = 1
)
SELECT start_year, intervention_type, COUNT(*) AS trials,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY start_year), 1) AS pct_of_year
FROM typed
GROUP BY start_year, intervention_type
ORDER BY start_year, trials DESC;
""",

"04a_top_sponsors_by_condition": """
WITH counts AS (
  SELECT c.condition_group, t.sponsor_name, t.sponsor_class,
         COUNT(DISTINCT t.nct_id) AS trials
  FROM trials t
  JOIN conditions c ON t.nct_id = c.nct_id
  WHERE c.condition_group <> 'Other'
  GROUP BY c.condition_group, t.sponsor_name, t.sponsor_class
),
ranked AS (
  SELECT *, RANK() OVER (PARTITION BY condition_group ORDER BY trials DESC) AS rnk
  FROM counts
)
SELECT * FROM ranked WHERE rnk <= 5
ORDER BY condition_group, rnk;
""",

"04b_sponsor_type_by_year": """
SELECT t.start_year, t.sponsor_class, COUNT(*) AS trials,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY t.start_year), 1) AS pct_of_year
FROM trials t
WHERE t.in_scope = 1
GROUP BY t.start_year, t.sponsor_class
ORDER BY t.start_year, trials DESC;
""",

"05a_stopped_early_by_sponsor": """
SELECT sponsor_class,
       COUNT(*) AS ended_trials,
       SUM(overall_status = 'COMPLETED') AS completed,
       SUM(overall_status IN ('TERMINATED','WITHDRAWN','SUSPENDED')) AS stopped_early,
       ROUND(100.0 * SUM(overall_status IN ('TERMINATED','WITHDRAWN','SUSPENDED'))
             / COUNT(*), 1) AS pct_stopped_early
FROM trials
WHERE in_scope = 1
  AND overall_status IN ('COMPLETED','TERMINATED','WITHDRAWN','SUSPENDED')
GROUP BY sponsor_class
ORDER BY ended_trials DESC;
""",

"05b_why_stopped_grouped": """
WITH reasons AS (
  SELECT sponsor_class, LOWER(why_stopped) AS r
  FROM trials
  WHERE in_scope = 1
    AND overall_status IN ('TERMINATED','WITHDRAWN','SUSPENDED')
    AND TRIM(COALESCE(why_stopped, '')) <> ''
),
grouped AS (
  SELECT sponsor_class,
    CASE
      WHEN r LIKE '%covid%' OR r LIKE '%pandemic%' OR r LIKE '%coronavirus%' THEN 'COVID-19'
      WHEN r LIKE '%efficacy%' OR r LIKE '%futility%' OR r LIKE '%interim analys%' THEN 'Efficacy / futility'
      WHEN r LIKE '%sponsor decision%' OR r LIKE '%business%' OR r LIKE '%strategic%'
        OR r LIKE '%portfolio%' OR r LIKE '%company decision%' OR r LIKE '%program%' THEN 'Sponsor / business decision'
      WHEN r LIKE '%fund%' OR r LIKE '%budget%' OR r LIKE '%financ%' THEN 'Funding'
      WHEN r LIKE '%recruit%' OR r LIKE '%enrol%' OR r LIKE '%accrual%'
        OR r LIKE '%participant%' OR r LIKE '%subject%' OR r LIKE '%patient%' THEN 'Recruitment'
      WHEN r LIKE '%pi %' OR r LIKE '%investigator%' OR r LIKE '%left the%'
        OR r LIKE '%staff%' OR r LIKE '%personnel%' THEN 'Investigator / staff'
      WHEN r LIKE '%safety%' OR r LIKE '%adverse%' THEN 'Safety'
      ELSE 'Other / unclear'
    END AS reason_group
  FROM reasons
)
SELECT reason_group,
       COUNT(*) AS trials,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct,
       SUM(sponsor_class = 'INDUSTRY') AS industry,
       SUM(sponsor_class = 'OTHER') AS academic
FROM grouped
GROUP BY reason_group
ORDER BY trials DESC;
""",

"06a_canadian_sites_by_condition": """
SELECT c.condition_group,
       COUNT(DISTINCT c.nct_id) AS all_trials,
       COUNT(DISTINCT CASE WHEN s.country = 'Canada' THEN c.nct_id END) AS with_canadian_site,
       ROUND(100.0 * COUNT(DISTINCT CASE WHEN s.country = 'Canada' THEN c.nct_id END)
             / COUNT(DISTINCT c.nct_id), 1) AS pct_canada
FROM conditions c
LEFT JOIN sites s ON c.nct_id = s.nct_id
WHERE c.condition_group <> 'Other'
GROUP BY c.condition_group
ORDER BY pct_canada DESC;
""",

"06b_top_countries": """
SELECT s.country, COUNT(DISTINCT t.nct_id) AS trials,
       ROUND(100.0 * COUNT(DISTINCT t.nct_id) / (SELECT COUNT(*) FROM trials WHERE in_scope = 1), 1) AS pct
FROM trials t JOIN sites s ON t.nct_id = s.nct_id
WHERE t.in_scope = 1
GROUP BY s.country ORDER BY trials DESC LIMIT 10;
""",

"07a_psychedelic_running_total": """
WITH yearly AS (
  SELECT t.start_year, i.psychedelic_class, COUNT(DISTINCT t.nct_id) AS trials
  FROM trials t
  JOIN interventions i ON t.nct_id = i.nct_id
  WHERE i.psychedelic_class IS NOT NULL AND t.in_scope = 1
  GROUP BY t.start_year, i.psychedelic_class
)
SELECT start_year, psychedelic_class, trials,
       SUM(trials) OVER (PARTITION BY psychedelic_class ORDER BY start_year) AS running_total
FROM yearly
ORDER BY psychedelic_class, start_year;
""",

"07b_psychedelic_sponsors_by_period": """
SELECT i.psychedelic_class,
       CASE WHEN t.start_year >= 2021 THEN '2021-2025' ELSE '2010-2020' END AS period,
       t.sponsor_class,
       COUNT(DISTINCT t.nct_id) AS trials
FROM trials t JOIN interventions i ON t.nct_id = i.nct_id
WHERE i.psychedelic_class IS NOT NULL AND t.in_scope = 1
GROUP BY 1, 2, 3
ORDER BY 1, 2, trials DESC;
""",
}

for name, sql in QUERIES.items():
    with open(f"{PROJECT}/sql/{name}.sql", "w") as f:
        f.write(sql.strip() + "\n")
    result = q(sql)
    result.to_csv(f"{PROJECT}/results/{name}.csv", index=False)
    print(f"{name}: {len(result)} rows")

## E. Validation

Compare my counts with searches on the ClinicalTrials.gov website (condition + Interventional + start year). Website results on 1 October 2026: schizophrenia 2020 = 121, PTSD 2022 = 149, eating disorder 2024 = 154.

In [ ]:
print(q("""
SELECT c.condition_group, t.start_year, COUNT(DISTINCT t.nct_id) AS my_count
FROM trials t JOIN conditions c ON t.nct_id = c.nct_id
WHERE (c.condition_group = 'Schizophrenia & psychosis' AND t.start_year = 2020)
   OR (c.condition_group = 'PTSD' AND t.start_year = 2022)
   OR (c.condition_group = 'Eating disorders' AND t.start_year = 2024)
GROUP BY 1, 2;
""").to_string())

Eating disorders: compare trial IDs with the website's 154 results to see which trials differ and why.

In [ ]:
# Website search results: Condition "eating disorder", Interventional, start date in 2024,
# downloaded as CSV from clinicaltrials.gov on 1 October 2026
import io
WEB_CSV = f"{PROJECT}/data/validation/ctgov_eating_disorder_2024.csv"
if os.path.exists(WEB_CSV):
    web = pd.read_csv(WEB_CSV)
else:
    from google.colab import files
    up = files.upload()
    web = pd.read_csv(io.BytesIO(list(up.values())[0]))
web_ids = set(web["NCT Number"])

my_ids = set(q("""
SELECT DISTINCT t.nct_id FROM trials t JOIN conditions c ON t.nct_id = c.nct_id
WHERE c.condition_group = 'Eating disorders' AND t.start_year = 2024
""")["nct_id"])

not_downloaded = web_ids - set(trials["nct_id"])
downloaded_not_grouped = (web_ids & set(trials["nct_id"])) - my_ids
print("On website:", len(web_ids), "| matched:", len(web_ids & my_ids),
      "| not in my download:", len(not_downloaded),
      "| downloaded but not grouped:", len(downloaded_not_grouped))

print(conditions[conditions.nct_id.isin(downloaded_not_grouped)]
      .groupby("condition")["nct_id"].nunique().sort_values(ascending=False).head(25).to_string())

What the trials missing from my download are about (mostly obesity, weight and diet, which the website links to "eating disorder" but which aren't eating disorders).

In [ ]:
missing = web[web["NCT Number"].isin(not_downloaded)]
print(missing["Conditions"].str.split("|").explode().str.strip()
      .value_counts().head(30).to_string())